# IBM AML Data Enrichment & MongoDB Ingestion
Notebook ini digunakan untuk memproses dataset **IBM Anti-Money Laundering (AML)** dari Kaggle, melakukan pengayaan data (enrichment) dengan metadata IP, Device, dan Kode BPS Wilayah Indonesia, serta mengekspornya ke format JSON/MongoDB.

In [ ]:
import os
import glob
import json
import time
import random
import numpy as np
import pandas as pd
from datetime import datetime

print("Library berhasil di-import.")

## 1. Memuat Dataset IBM dari Kaggle Input

In [ ]:
# Mencari lokasi file IBM di direktori Kaggle secara otomatis
trans_file = None
acc_file = None

for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        if file == 'HI-Medium_Trans.csv':
            trans_file = os.path.join(root, file)
        elif file == 'HI-Medium_accounts.csv':
            acc_file = os.path.join(root, file)

print(f"File Transaksi ditemukan di: {trans_file}")
print(f"File Akun ditemukan di: {acc_file}")

# Membaca dataset (Ambil 50.000 sampel agar cepat, hilangkan nrows=50000 jika ingin semua data)
df_trans = pd.read_csv(trans_file, nrows=50000)
df_accounts = pd.read_csv(acc_file)

print(f"Jumlah Transaksi dimuat: {len(df_trans)}")
print(f"Jumlah Akun dimuat: {len(df_accounts)}")

## 2. Pemetaan & Standarisasi Kolom ke MongoDB `dataset_transaksi`

In [ ]:
columns_mapping = {}
acc_count = 0
for col in df_trans.columns:
    if col == 'Timestamp':
        columns_mapping[col] = 'timestamp'
    elif col == 'From Bank':
        columns_mapping[col] = 'sender_bank'
    elif col == 'To Bank':
        columns_mapping[col] = 'receiver_bank'
    elif col == 'Account':
        if acc_count == 0:
            columns_mapping[col] = 'sender_account'
            acc_count += 1
        else:
            columns_mapping[col] = 'receiver_account'
    elif col.startswith('Account'):
        columns_mapping[col] = 'receiver_account'
    elif col == 'Amount Received':
        columns_mapping[col] = 'amount_received'
    elif col == 'Receiving Currency':
        columns_mapping[col] = 'receiving_currency'
    elif col == 'Amount Paid':
        columns_mapping[col] = 'amount_paid'
    elif col == 'Payment Currency':
        columns_mapping[col] = 'payment_currency'
    elif col == 'Payment Format':
        columns_mapping[col] = 'payment_format'
    elif col == 'Is Laundering':
        columns_mapping[col] = 'is_laundering'

df_trans = df_trans.rename(columns=columns_mapping)
print("Kolom Disesuaikan:", df_trans.columns.tolist())

## 3. Proses Enrichment Data (IP, Device, Lokasi Indonesia & Kode BPS)

In [ ]:
indonesia_regions = [
    {"location": "Jakarta Selatan", "bps_code": "3174"},
    {"location": "Jakarta Timur", "bps_code": "3172"},
    {"location": "Surabaya", "bps_code": "3578"},
    {"location": "Bandung", "bps_code": "3273"},
    {"location": "Medan", "bps_code": "1275"},
    {"location": "Semarang", "bps_code": "3374"},
    {"location": "Makassar", "bps_code": "7371"},
    {"location": "Tangerang Selatan", "bps_code": "3674"},
    {"location": "Depok", "bps_code": "3276"},
    {"location": "Batam", "bps_code": "2171"}
]

devices_list = ["Android-Mobile", "iOS-Mobile", "Web-Chrome", "Web-Firefox", "ATM-Terminal"]

def generate_ip():
    return f"{random.randint(1, 223)}.{random.randint(0, 255)}.{random.randint(0, 255)}.{random.randint(1, 254)}"

def parse_timestamp(ts_str):
    try:
        dt = datetime.strptime(str(ts_str), "%Y/%m/%d %H:%M")
        return int(dt.timestamp() * 1000)
    except:
        return int(time.time() * 1000)

# 1. Konversi timestamp ke epoch milidetik
df_trans['timestamp'] = df_trans['timestamp'].apply(parse_timestamp)

# 2. Tambahkan metadata IP, Device, Lokasi, dan Status
np.random.seed(42)
num_rows = len(df_trans)

df_trans['ip'] = [generate_ip() for _ in range(num_rows)]
df_trans['device'] = np.random.choice(devices_list, size=num_rows)

chosen_regions = np.random.choice(indonesia_regions, size=num_rows)
df_trans['location'] = [r['location'] for r in chosen_regions]
df_trans['bps_code'] = [r['bps_code'] for r in chosen_regions]

df_trans['prediction_status'] = 'processed'
df_trans['created_at'] = datetime.utcnow().isoformat() + "Z"

print("Enrichment berhasil dilakukan!")

## 4. Join dengan Master Account (`HI-Medium_accounts.csv`)

In [ ]:
df_accounts_clean = df_accounts.rename(columns={
    'Account No.': 'sender_account',
    'Entity Name': 'sender_entity_name',
    'Bank Name': 'sender_bank_name'
})[['sender_account', 'sender_entity_name', 'sender_bank_name']]

df_enriched = pd.merge(df_trans, df_accounts_clean, on='sender_account', how='left')
df_enriched['sender_entity_name'] = df_enriched['sender_entity_name'].fillna('Individual Account')
df_enriched['sender_bank_name'] = df_enriched['sender_bank_name'].fillna('Unknown Bank')

df_enriched.head()

## 5. Ekspor Hasil ke JSON & CSV

In [ ]:
output_json_path = 'dataset_transaksi_enriched.json'
output_csv_path = 'dataset_transaksi_enriched.csv'

records = df_enriched.to_dict(orient='records')
with open(output_json_path, 'w') as f:
    for record in records:
        f.write(json.dumps(record) + '\n')

df_enriched.to_csv(output_csv_path, index=False)

print(f"File JSON berhasil dibuat: {output_json_path}")
print(f"File CSV berhasil dibuat: {output_csv_path}")

## 6. (Opsional) Auto-Upload Direct ke MongoDB Atlas

In [ ]:
!pip install -q pymongo dnspython

from pymongo import MongoClient

MONGO_URI = "YOUR_MONGODB_ATLAS_CONNECTION_STRING"

if MONGO_URI != "YOUR_MONGODB_ATLAS_CONNECTION_STRING":
    client = MongoClient(MONGO_URI)
    db = client['fraud_detection']
    collection = db['dataset_transaksi']
    
    batch_size = 5000
    total = len(records)
    for i in range(0, total, batch_size):
        batch = records[i:i+batch_size]
        collection.insert_many(batch)
        print(f"Uploaded batch {i} - {min(i+batch_size, total)}")
    print("Upload MongoDB Atlas Selesai!")
else:
    print("Silakan isi MONGO_URI jika ingin upload langsung ke Atlas.")